In [1]:
! pip install datasets
! pip install evaluate
! pip install sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 1.5 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 1.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.0/104.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 37.5 MB/s eta 0:00:00a 0:00:01


In [2]:
import numpy as np
import torch
import torch.nn as nn
import evaluate
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader

from transformers import T5Tokenizer, T5ForConditionalGeneration

In [ ]:
#%load_ext blackcellmagic

In [3]:
CHECKPOINT = "google-t5/t5-small"
DATASET = "Helsinki-NLP/opus_books"

In [4]:
tokenizer = T5Tokenizer.from_pretrained(CHECKPOINT)
model = T5ForConditionalGeneration.from_pretrained(CHECKPOINT)

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [52]:
# load dataset
dataset = load_dataset(DATASET, "en-fr")['train']

# shuffle the dataset
dataset = dataset.shuffle(seed=42)
print(len(dataset))
# Split the dataset into train, valid, test(train_size=80%, valid_size=10%, test_size=10%)
train_valid_dataset, test_dataset = torch.utils.data.random_split(dataset, [0.9, 0.1])
train_dataset, valid_dataset = torch.utils.data.random_split(train_valid_dataset, [0.9, 0.1])

test_dataset, _ = torch.utils.data.random_split(test_dataset, [0.2, 0.8]) # Further reduce the size since otherwise computations are slow

127085


In [53]:
len(test_dataset)

2542

In [7]:
class PrefixLayer(nn.Module):
    def __init__(self, prefix_length, hidden_size, prefix_projection_size):
        super().__init__()
        self.prefix_length = prefix_length  # |Pidx|
        self.hidden_size = hidden_size  # dim(hi)
        self.prefix_projection_size = prefix_projection_size  # 512 for table-to-text and 800 for summarization

        # Initialize the smaller prefix parameters (P'θ)
        self.prefix_projection = nn.Parameter(
            torch.randn(self.prefix_length, self.prefix_projection_size)
        )

        # MLP for reparametrization
        self.prefix_mlp = nn.Sequential(
            nn.Linear(self.prefix_projection_size, self.hidden_size),
            nn.Tanh(),
            nn.Linear(self.hidden_size, self.hidden_size),
        )

    def forward(self):
        prefix_tokens =  self.prefix_projection  # shape: (prefix_length, prefix_projection_size)
        prefix = self.prefix_mlp(prefix_tokens)  # shape: (prefix_length, hidden_size)
        return prefix

In [8]:
class PrefixT5(nn.Module):

    def __init__(self, t5_model, prefix_length, prefix_projection_size=512):
        super().__init__()
        self.t5 = t5_model
        self.hidden_size = self.t5.config.hidden_size
        self.prefix_length = prefix_length
        self.encoder_prefix = PrefixLayer(self.prefix_length, self.hidden_size, prefix_projection_size)
        self.decoder_prefix = PrefixLayer(self.prefix_length, self.hidden_size, prefix_projection_size)

        # Freeze the base model parameters
        for param in self.t5.parameters():
            param.requires_grad = False

    def forward(self, input_ids, decoder_input_ids, labels, attention_mask=None):
        batch_size = input_ids.shape[0]

        # prepend the prefix to the encoder embeddings
        encoder_prefix = self.encoder_prefix().unsqueeze(0).expand(batch_size, -1, -1)
        encoder_embeddings = self.t5.shared(input_ids)                                            # shape: (batch_size, seq_len, hidden_size)
        concatenated_encoder_embeddings = torch.cat([encoder_prefix, encoder_embeddings], dim=1)  # shape: (batch_size, prefix_len+seq_len, hidden_size)

        # prepend the prefix to the decoder embeddings
        decoder_prefix = self.decoder_prefix().unsqueeze(0).expand(batch_size, -1, -1)
        decoder_embeddings = self.t5.shared(decoder_input_ids)
        concatenated_decoder_embeddings = torch.cat([decoder_prefix, decoder_embeddings], dim=1)

        # encoder attention mask
        prefix_attention = torch.ones(batch_size, self.prefix_length, device=input_ids.device)
        encoder_attention_mask = torch.cat([prefix_attention, attention_mask], dim=1)

        encoder_outputs = self.t5.encoder(
            inputs_embeds=concatenated_encoder_embeddings,
            attention_mask=encoder_attention_mask,
            return_dict=True
        )

        if labels is not None:
            # Ignore prefix positions in loss calculation
            new_labels = torch.full(
                    (batch_size, self.prefix_length),
                    -100,
                    device=labels.device,
                    dtype=labels.dtype
            )
            labels = torch.cat([new_labels, labels], dim=1)
        else:
            labels = None    # during generation, the labels are None

        return self.t5(
            encoder_outputs=encoder_outputs,
            decoder_inputs_embeds=concatenated_decoder_embeddings,
            labels=labels
        )



In [64]:
metric = evaluate.load("sacrebleu")

In [ ]:
def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [[label.strip()] for label in labels]

    return preds, labels


def compute_metrics(eval_preds):
    preds, labels = eval_preds

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds, decoded_labels = postprocess_text(preds, labels)

    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    result = {"bleu": result["score"]}

    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in preds]
    result["gen_len"] = np.mean(prediction_lens)
    result = {k: round(v, 4) for k, v in result.items()}
    return result

In [ ]:
@torch.no_grad()
def generate_translation(
    model: torch.nn.Module,
    tokenizer: T5Tokenizer,
    input_text, max_length: int = 128,
    device: str = "cuda" if torch.cuda.is_available() else "cpu",
) -> str:

    """Generate and print sample translations"""
    model.eval().to(device)

    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, padding=True).to(device)
    input_ids = inputs["input_ids"]
    attention_mask = inputs["attention_mask"]

    decoder_input_ids = torch.full(
        (input_ids.size(0), 1),  # batch_size x 1
        tokenizer.pad_token_id,  # Start with <pad> (can be adjusted for specific tasks)
        device=device,
    )

    generated_ids = []

    for _ in range(max_length):

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=None,
            decoder_input_ids=decoder_input_ids,
        )

        logits = outputs.logits[:, -1, :]
        next_token_id = logits.argmax(dim=-1)
        generated_ids.append(next_token_id)

        decoder_input_ids = torch.cat([decoder_input_ids, next_token_id.unsqueeze(-1)], dim=-1)

        if next_token_id.item() == tokenizer.eos_token_id:
            break

    generated_text = tokenizer.decode(torch.cat(generated_ids), skip_special_tokens=True)
    return generated_text

In [68]:
text, reference = test_dataset[0]['translation'].values()
print(text)
outputs = generate_translation(
    model=prefixed_t5_model,
    tokenizer=tokenizer,
    input_text='translate English to French: ' + text,
    max_length=1000,
)
reference, outputs

"You comprehend?" said Porthos.


('-- Vous comprenez? dit Porthos.', '« Tu comprends? », a dit Porthos.')

In [70]:
from tqdm import tqdm

In [80]:
references = [example['fr'] for example in test_dataset[:]['translation']]
predictions = [
                generate_translation(model=model,
                    tokenizer=tokenizer,
                    input_text='translate English to French: ' + example['en'],
                    max_length=1000
                )
                for example in tqdm(test_dataset[:]['translation'], total=len(test_dataset[:]['translation']))
            ]
compute_metrics((predictions, references))

{'bleu': 15.1709, 'gen_len': 1.0}

In [96]:
sample = "translate English to French: Hello, how are you?"
target_text = "Bonjour, comment ça va ?"


encoded = tokenizer(
    text=sample,
    text_target=target_text,
    padding=True,
    truncation=True,
    return_tensors="pt",
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

input_ids = encoded["input_ids"].to(device)
labels = encoded["labels"].to(device)
attention_mask = encoded["attention_mask"].to(device)
decoder_input_ids = model._shift_right(labels).to(device)

prefixed_t5_model = PrefixT5(
    t5_model=model, prefix_length=5, prefix_projection_size=512
).to(device)
checkpoint = torch.load("../../root/models/prefix_len_5-proj_size_512-lr_5e-05.pth")
prefixed_t5_model.load_state_dict(checkpoint["model_state_dict"])

outputs = prefixed_t5_model(
    input_ids=input_ids,
    decoder_input_ids=decoder_input_ids,
    labels=labels,
    attention_mask=attention_mask,
)

print("Loss:", outputs.loss)
print("Logits Shape:", outputs.logits.shape)

Loss: tensor(2.2379, device='cuda:0', grad_fn=<NllLossBackward0>)
Logits Shape: torch.Size([1, 13, 32128])


/tmp/ipykernel_122548/3579961909.py:22: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load("../../root/models/prefix_len_5-proj_size_512-lr_5e-05.pth")


In [97]:
# prefix_len_5-proj_size_512-lr_5e-05.pth
references = [example['fr'] for example in test_dataset[:]['translation']]
predictions = [
                generate_translation(model=prefixed_t5_model,
                    tokenizer=tokenizer,
                    input_text='translate English to French: ' + example['en'],
                    max_length=1000
                )
                  for example in tqdm(test_dataset[:]['translation'], total=len(test_dataset[:]['translation']))
]

compute_metrics((predictions, references))

100%|██████████| 2542/2542 [36:03<00:00,  1.18it/s]  


{'bleu': 13.414, 'gen_len': 1.0}

In [84]:
# prefix_len_5-proj_size_512-lr_8e-05.pth
references = [example['fr'] for example in test_dataset[:]['translation']]
predictions = [
                generate_translation(model=prefixed_t5_model,
                    tokenizer=tokenizer,
                    input_text='translate English to French: ' + example['en'],
                    max_length=1000
                )
                  for example in tqdm(test_dataset[:]['translation'], total=len(test_dataset[:]['translation']))
]

compute_metrics((predictions, references))

  4%|▎         | 93/2542 [2:38:42<69:39:25, 102.40s/it]


100%|██████████| 2542/2542 [44:30<00:00,  1.05s/it]  


{'bleu': 10.8916, 'gen_len': 1.0}

In [88]:
# prefix_len_10-proj_size_512-lr_5e-05.pth
references = [example['fr'] for example in test_dataset[:]['translation']]
predictions = [
                generate_translation(model=prefixed_t5_model,
                    tokenizer=tokenizer,
                    input_text='translate English to French: ' + example['en'],
                    max_length=1000
                )
                  for example in tqdm(test_dataset[:]['translation'], total=len(test_dataset[:]['translation']))
]

compute_metrics((predictions, references))

  0%|          | 0/2542 [00:00<?, ?it/s]

100%|██████████| 2542/2542 [35:47<00:00,  1.18it/s]  


{'bleu': 12.3649, 'gen_len': 1.0}

In [90]:
# prefix_len_10-proj_size_512-lr_8e-05.pth
references = [example['fr'] for example in test_dataset[:]['translation']]
predictions = [
                generate_translation(model=prefixed_t5_model,
                    tokenizer=tokenizer,
                    input_text='translate English to French: ' + example['en'],
                    max_length=1000
                )
                  for example in tqdm(test_dataset[:]['translation'], total=len(test_dataset[:]['translation']))
]

compute_metrics((predictions, references))

  0%|          | 7/2542 [5:18:39<1923:17:34, 2731.30s/it]


{'bleu': 10.8003, 'gen_len': 1.0}

In [93]:
# prefix_len_15-proj_size_512-lr_5e-05.pth
references = [example['fr'] for example in test_dataset[:]['translation']]
predictions = [
                generate_translation(model=prefixed_t5_model,
                    tokenizer=tokenizer,
                    input_text='translate English to French: ' + example['en'],
                    max_length=1000
                )
                  for example in tqdm(test_dataset[:]['translation'], total=len(test_dataset[:]['translation']))
]

compute_metrics((predictions, references))

100%|██████████| 2542/2542 [39:34<00:00,  1.07it/s]  


{'bleu': 12.2618, 'gen_len': 1.0}

In [95]:
# prefix_len_15-proj_size_512-lr_8e-05.pth
references = [example['fr'] for example in test_dataset[:]['translation']]
predictions = [
                generate_translation(model=prefixed_t5_model,
                    tokenizer=tokenizer,
                    input_text='translate English to French: ' + example['en'],
                    max_length=1000
                )
                  for example in tqdm(test_dataset[:]['translation'], total=len(test_dataset[:]['translation']))
]

compute_metrics((predictions, references))

100%|██████████| 2542/2542 [37:19<00:00,  1.14it/s]  


{'bleu': 12.7718, 'gen_len': 1.0}